# EDA

## Table of contents

1. [Data loading](#data-loading)
   - [Read parquet files](#read-parquet-files)
   - [Enrich with document metadata](#enrich-with-document-metadata)
2. [Descriptive statistics](#descriptive-statistics)
   - [Partition size](#partition-size)
   - [Document metrics](#document-metrics)
   - [Label distribution](#label-distribution)
3. [Text source comparison](#text-source-comparison)
   - [Length comparison](#length-comparison)
   - [Top divergent documents](#top-divergent-documents-train)

In [ ]:
import re
import subprocess
from pathlib import Path
from typing import Literal

import altair as alt
import pandas as pd
import pymupdf

type Partition = Literal["train", "dev-0", "test-A"]

In [ ]:
static_path = Path().cwd() / "src" / "nda" / "static"
outputs_path = static_path / "outputs"

## Data loading

### Read parquet files

In [ ]:
def read_parquet_file(partition: Partition) -> pd.DataFrame:
    file_path = outputs_path / partition / "data.parquet"

    if not file_path.exists():
        raise FileNotFoundError(f"Parquet file not found at {file_path}")

    if partition == "test-A":
        return pd.read_parquet(file_path).loc[
            :,
            [
                "filename",
                "text_best",
            ],
        ]
    return pd.read_parquet(file_path).loc[
        :,
        [
            "filename",
            "labels",
            "labels_schema",
            "text_best",
        ],
    ]

In [ ]:
train, val, test = [
    read_parquet_file(partition) for partition in ("train", "dev-0", "test-A")
]

train.head()

### Enrich with document metadata

In [ ]:
def get_document_metadata(documents_path: Path, filename: str) -> dict:
    with pymupdf.open(documents_path / filename) as doc:
        page_count = doc.page_count
        full_text = "\n".join(str(page.get_text("text")) for page in doc)
        full_text = re.sub(r"\xa0+", " ", full_text)
        full_text = re.sub(r"\n \n", "\n", full_text)

    return {
        "filename": filename,
        "text_pymupdf": full_text,
        "char_count": len(full_text),
        "word_count": len(re.split(r"\s+", full_text.strip())),
        "line_count": full_text.count("\n") + 1,
        "page_count": page_count,
    }

In [ ]:
def enrich_with_document_metadata(
    df: pd.DataFrame, partition: Partition
) -> pd.DataFrame:
    documents_path = outputs_path / partition / "documents"
    metadata = [
        get_document_metadata(documents_path, filename) for filename in df["filename"]
    ]
    return df.merge(pd.DataFrame(metadata), on="filename", how="left")

In [ ]:
train, val, test = [
    enrich_with_document_metadata(df, partition)
    for df, partition in zip(
        (train, val, test), ("train", "dev-0", "test-A"), strict=True
    )
]

train.head()

## Descriptive statistics

### Partition size 

In [ ]:
print(
    f"Train set: {len(train)} samples\n"
    f"Validation set: {len(val)} samples\n"
    f"Test set: {len(test)} samples"
)

### Document metrics

In [ ]:
def check_distributions(df: pd.DataFrame, partition: Partition) -> alt.HConcatChart:
    metrics = ["char_count", "word_count", "line_count", "page_count"]

    charts = [
        alt.Chart(df[[metric]])
        .mark_bar()
        .encode(
            x=alt.X(f"{metric}:Q", bin=alt.Bin(maxbins=30), title=metric),
            y=alt.Y("count()", title="count"),
        )
        .properties(title=metric, width=200, height=200)
        for metric in metrics
    ]

    return alt.hconcat(*charts).properties(
        title=alt.TitleParams(
            f"Distributions of document metrics for '{partition}' partition", offset=20
        )
    )


def fetch_statistics(df: pd.DataFrame) -> pd.DataFrame:
    statistics = ["mean", "median", "min", "max", "kurtosis", "skew"]
    metrics = ["char_count", "word_count", "line_count", "page_count"]
    return df.agg({metric: statistics for metric in metrics})

In [ ]:
alt.vconcat(
    check_distributions(train, "train"),
    check_distributions(val, "dev-0"),
    check_distributions(test, "test-A"),
).properties(
    padding={"left": 30, "right": 30, "top": 30, "bottom": 30},
)

In [ ]:
for df, partition in zip((train, val, test), ("train", "dev-0", "test-A"), strict=True):
    print(f"Statistics for '{partition}' partition:")
    display(fetch_statistics(df))

### Label distribution

#### Null rate per label

In [ ]:
def plot_label_null_rates(df: pd.DataFrame, partition: Partition) -> alt.Chart:
    labels_df = pd.DataFrame(df["labels_schema"].tolist())
    null_rates = labels_df.isnull().mean().reset_index()
    null_rates.columns = ["label", "null_rate"]

    return (
        alt.Chart(null_rates)
        .mark_bar()
        .encode(
            x=alt.X("null_rate:Q", title="null rate", scale=alt.Scale(domain=[0, 1])),
            y=alt.Y("label:N", title=None, sort="-x"),
            tooltip=[
                alt.Tooltip("label:N"),
                alt.Tooltip("null_rate:Q", format=".1%"),
            ],
        )
        .properties(title=partition, width=250, height=150)
    )

In [ ]:
alt.hconcat(
    plot_label_null_rates(train, "train"),
    plot_label_null_rates(val, "dev-0"),
).properties(
    title=alt.TitleParams("Null rate per label", offset=20),
    padding={"left": 30, "right": 30, "top": 30, "bottom": 30},
)

#### Value distributions

In [ ]:
def plot_label_value_distributions(
    df: pd.DataFrame, partition: Partition, top_n: int = 10
) -> alt.HConcatChart:
    labels_df = pd.DataFrame(df["labels_schema"].tolist())

    charts = [
        alt.Chart(
            labels_df[label]
            .dropna()
            .explode()
            .apply(lambda x: x["name"] if isinstance(x, dict) else x)
            .value_counts()
            .head(top_n)
            .reset_index()
            .rename(columns={label: "value"})
        )
        .mark_bar()
        .encode(
            x=alt.X("count:Q", title="count"),
            y=alt.Y("value:N", title=None, sort="-x"),
            tooltip=[
                alt.Tooltip("value:N"),
                alt.Tooltip("count:Q"),
            ],
        )
        .properties(title=label, width=200, height=150)
        for label in labels_df.columns
    ]

    return alt.hconcat(*charts).properties(
        title=alt.TitleParams(
            f"Top {top_n} value distributions per label for '{partition}' partition",
            offset=20,
        )
    )

In [ ]:
alt.vconcat(
    plot_label_value_distributions(train, "train"),
    plot_label_value_distributions(val, "dev-0"),
).properties(
    padding={"left": 30, "right": 30, "top": 30, "bottom": 30},
)

## Text source comparison

In [ ]:
def compute_text_comparison(df: pd.DataFrame) -> pd.DataFrame:
    df_text = df[["filename", "text_best", "text_pymupdf"]].copy()

    return (
        df_text.assign(len_best=lambda x: x["text_best"].str.len())
        .assign(len_pymupdf=lambda x: x["text_pymupdf"].str.len())
        .assign(char_delta=lambda x: x["len_best"] - x["len_pymupdf"])
        .assign(relative_delta=lambda x: x["char_delta"] / x["len_pymupdf"])
    )

In [ ]:
cmp_train, cmp_val, cmp_test = [
    compute_text_comparison(df) for df in (train, val, test)
]

cmp_train.head()

### Length comparison

In [ ]:
def plot_length_scatter(df: pd.DataFrame, partition: Partition) -> alt.LayerChart:
    max_val = max(df["len_best"].max(), df["len_pymupdf"].max())
    diagonal = pd.DataFrame({"x": [0, max_val], "y": [0, max_val]})

    scatter = (
        alt.Chart(df)
        .mark_circle(opacity=0.5, size=30)
        .encode(
            x=alt.X("len_pymupdf:Q", title="len(text_pymupdf)"),
            y=alt.Y("len_best:Q", title="len(text_best)"),
            tooltip=[
                alt.Tooltip("filename:N"),
                alt.Tooltip("len_best:Q"),
                alt.Tooltip("len_pymupdf:Q"),
                alt.Tooltip("char_delta:Q"),
            ],
        )
    )

    identity = (
        alt.Chart(diagonal)
        .mark_line(color="red", strokeDash=[4, 4], opacity=0.6)
        .encode(x="x:Q", y="y:Q")
    )

    return (scatter + identity).properties(title=partition, width=240, height=240)


alt.hconcat(
    plot_length_scatter(cmp_train, "train"),
    plot_length_scatter(cmp_val, "dev-0"),
    plot_length_scatter(cmp_test, "test-A"),
).properties(
    title=alt.TitleParams(
        "Length comparison: text_best vs text_pymupdf",
        offset=20,
    ),
    padding={"left": 30, "right": 30, "top": 30, "bottom": 30},
)

### Top divergent documents per partition

In [ ]:
def display_top_divergent_documents(
    df: pd.DataFrame, partition: Partition, top_n: int = 5
) -> None:
    display_cols = [
        "filename",
        "len_best",
        "len_pymupdf",
        "char_delta",
        "relative_delta",
    ]

    top_positive = df.nlargest(top_n, "relative_delta")[display_cols]
    top_negative = df.nsmallest(top_n, "relative_delta")[display_cols]

    for subset_df, label in zip(
        (top_positive, top_negative), ("positive", "negative"), strict=True
    ):
        print(f"Top {len(subset_df)} {label} deltas for '{partition}' partition:")
        display(subset_df)

In [ ]:
for df, partition in zip(
    (cmp_train, cmp_val, cmp_test), ("train", "dev-0", "test-A"), strict=True
):
    display_top_divergent_documents(df, partition)
    print("\n" + "=" * 89 + "\n")

#### Manual checking

In [ ]:
def open_document(filename: str) -> None:
    pdf_path = static_path / "data" / "documents" / f"{filename}"
    subprocess.run(["open", pdf_path])


open_document("724a6c9ea428637bd128cafb401b8a7e.pdf")

## Next steps

- Token counts approximation with tiktoken
- Manual inspection for outliers
- Jaccard similarity
- Conclusions section